<h1>Librairies</h1>

In [1]:
!pip install evaluate sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.8 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments, Trainer, AutoModel
import evaluate
import re

2026-02-03 22:49:40.694336: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770158980.979281      25 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770158981.078779      25 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770158981.805982      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770158981.806029      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770158981.806032      25 computation_placer.cc:177] computation placer alr

In [3]:
data_pd = pd.read_csv('/kaggle/input/deep-past-initiative-machine-translation/train.csv')
data_pd

,oare_id,transliteration,translation
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ..."
1,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...
2,0073f2c0-524c-4bbf-915a-8c1772a4fb98,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,... he did not give you a textile. He returned...
3,009fb838-8038-42bc-ad34-5f795b3840ee,KIŠIB šu-(d)EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū..."
4,00aa1c55-c80c-4346-a159-73ad43ab0ff7,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...
...,...,...,...
1556,ff3208e4-8ab8-4368-b4df-7b80afa5bc32,um-ma en-nam-a-šur-ma a-na a-la-ḫi-im qí-bi-ma...,From Ennam-Aššur to Ali-ahum: Here 2 men have ...
1557,ff43a284-3d67-4238-8b4a-9b6cb7531e0a,3 ma-na KÙ.BABBAR ṣa-ru-pá-am i-na ší-im SÍG.Ḫ...,Ilī-ašrannī son of Sukkalliya has received 3 m...
1558,ff5747a4-af8a-4100-a906-a2660ae72606,ša-lim-a-šùr a-na a-mur-IŠTAR ú-ṭá-ḫi-ni-a-tí-...,Šalim-Aššur made us approach Amur-Ištar and Ša...
1559,ff777871-97ce-4bfc-bdfb-73352868944d,a-na en-nam-a-šùr qí-bi-ma um-ma IŠTAR-ra-bi₄-...,To Ennam-Aššur from Ištar-rabiʾat: With respec...


In [4]:
print(data_pd.iloc[:,1:].values)

[['KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠIB šu-(d)EN.LÍL DUMU ma-nu-ki-a-šur KIŠIB MAN-a-šur DUMU a-ta-a 0.33333 ma-na 2 GÍN KÙ.BABBAR SIG₅ i-ṣé-er PUZUR₄-a-šur DUMU a-ta-a a-lá-ḫu-um i-šu iš-tù ḫa-muš-tim ša ì-lí-dan ITU.KAM ša ke-na-tim li-mu-um e-na-sú-in a-na ITU 14 ḫa-am-ša-tim i-ša-qal šu-ma lá iš-qú-ul 1.5 GÍN.TA a-na 1 ma-na-im i-na ITU.1.KAM ṣí-ib-tám ú-ṣa-áb'
  'Seal of Mannum-balum-Aššur son of Ṣilli-Adad, seal of Šu-Illil son of Mannum-kī-Aššur, seal of Puzur-Aššur son of Ataya. Puzur-Aššur son of Ataya owes 22 shekels of good silver to Ali-ahum. Reckoned from the week of Ilī-dan, month of Ša-kēnātim, in the eponymy of Enna-Suen, he will pay in 14 weeks. If he has not paid in time, he will add interest at the rate 1.5 shekel per mina per month.']
 ['1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé'
  'Itūr-ilī has received one textile of ordinary quality.']
 ['TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR'
  '... he did not give you a textile. He returned and 9 shekels of si

In [5]:
#model_name = "/kaggle/input/byt5-base/transformers/default/1/byt5-base-saved"  # or byt5-large
model_name = "/kaggle/input/byt5-first-10-epochs/transformers/default/6/byt5-akkadian/checkpoint-8800"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [6]:
def preprocess_akkadian_text(text, is_translation=False):
    """
    Clean Akkadian transliteration or translation text for LLM training.
    
    Parameters:
    -----------
    text : str
        The raw Akkadian transliteration or translation text
    is_translation : bool
        If True, applies translation-specific cleaning
    
    Returns:
    --------
    str : Cleaned text ready for LLM training
    """
    
    # Handle special Unicode characters (convert to standard form)
    # Normalize special Akkadian characters
    char_replacements = {
        'á': 'a₂', 'à': 'a₃',
        'é': 'e₂', 'è': 'e₃',
        'í': 'i₂', 'ì': 'i₃',
        'ú': 'u₂', 'ù': 'u₃',
        'š': 'š', 'Š': 'Š',  # Keep š/Š as they are proper Unicode
        'Ṣ': 'Ṣ', 'ṣ': 'ṣ',  # Keep as proper Unicode
        'Ṭ': 'Ṭ', 'ṭ': 'ṭ',  # Keep as proper Unicode
        'Ḫ': 'H', 'ḫ': 'h',  # Convert Ḫ/ḫ to H/h as per instructions
    }
    
    for old, new in char_replacements.items():
        text = text.replace(old, new)
    
    # Handle subscript numbers - remove subscript formatting but keep numbers
    # Convert subscript numbers to regular numbers (₀-₉ → 0-9)
    subscript_to_normal = {
        '₀': '0', '₁': '1', '₂': '2', '₃': '3', '₄': '4',
        '₅': '5', '₆': '6', '₇': '7', '₈': '8', '₉': '9',
        'ₓ': 'x'  # Special subscript x
    }
    
    for sub, normal in subscript_to_normal.items():
        text = text.replace(sub, normal)
    
    # REMOVE modern scribal notations (transliteration specific)
    if not is_translation:
        # Remove: ! ? / : .
        text = re.sub(r'[!?/:.]', '', text)
        
        # Remove partially broken signs ˹ ˺
        text = text.replace('˹', '').replace('˺', '')
        
        # Remove content in square brackets but keep the text inside
        # [KÙ.BABBAR] → KÙ.BABBAR
        text = re.sub(r'\[([^\]]*)\]', r'\1', text)
        
        # Remove parentheses but keep content inside (for translations, we handle differently)
        text = re.sub(r'\(([^)]*)\)', r'\1', text)
        
        # Remove scribal insertions markers but keep the text
        text = re.sub(r'<([^>]*)>', r'\1', text)
        
        # Remove double pointy brackets (erroneous signs)
        text = re.sub(r'<<([^>]*)>>', r'\1', text)
    
    # REPLACE breaks and gaps
    # Single sign break
    text = re.sub(r'\[x\]|x{1,3}|\bxx\b', ' <gap> ', text)
    
    # Large/multiple breaks
    text = re.sub(r'\[\.\.\.\]|\.\.\.|\[…\]|…|\[\.\.\. \.\.\.\]', ' <big_gap> ', text)
    
    # Handle determinatives in curly brackets
    # Replace determinative markers with clean format
    determinatives = {
        r'\{d\}': 'DINGIR',  # god/deity
        r'\{mul\}': 'MUL',  # stars
        r'\{ki\}': 'KI',  # earth/place
        r'\{lu[₂2]?\}': 'LU',  # people/professions
        r'\{e[₂2]?\}': 'É',  # buildings
        r'\{uru\}': 'URU',  # settlements
        r'\{kur\}': 'KUR',  # lands/mountains
        r'\{mi\}': 'MUNUS',  # feminine
        r'\{m\}': 'M',  # masculine
        r'\{geš\}|\{ĝeš\}': 'GIŠ',  # wood/trees
        r'\{tug[₂2]?\}': 'TÚG',  # textiles
        r'\{dub\}': 'DUB',  # tablets/documents
        r'\{id[₂2]?\}': 'ÍD',  # canals/rivers
        r'\{mušen\}': 'MUŠEN',  # birds
        r'\{na[₄4]?\}': 'NA₄',  # stone
        r'\{kuš\}': 'KUŠ',  # skins/hides
        r'\{u[₂2]?\}': 'Ú',  # plants
    }
    
    for pattern, replacement in determinatives.items():
        text = re.sub(pattern, replacement, text)
    
    # Clean up determinatives that might appear with words
    # Example: a-lim{ki} → a-lim KI
    text = re.sub(r'(\w+)\{([^}]+)\}', r'\1 \2', text)
    
    # Handle line numbers and apostrophes
    # Remove line numbers like 1, 5, 10, 15' , 20'' etc.
    text = re.sub(r'\b\d+\'*\b', '', text)
    
    # Handle subscripted vowels in determinatives
    # Remove curly brackets from any remaining determinatives
    text = text.replace('{', '').replace('}', '')
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Special handling for translation text
    if is_translation:
        # Keep parentheses in translations as they might contain useful info
        # But remove if they're empty or just contain scribal notes
        text = re.sub(r'\(\s*\)', '', text)
        # Remove excessive punctuation in translations
        text = re.sub(r'[!?;:]+', '.', text)
    
    # Final cleanup
    text = text.replace('  ', ' ').strip()
    
    return text


def preprocess_dataset_entry(transliteration, translation):
    """
    Preprocess a complete dataset entry (transliteration + translation).
    
    Parameters:
    -----------
    transliteration : str
        Raw Akkadian transliteration text
    translation : str
        Raw English translation text
    
    Returns:
    --------
    tuple : (cleaned_transliteration, cleaned_translation)
    """
    clean_translit = preprocess_akkadian_text(transliteration, is_translation=False)
    clean_translation = preprocess_akkadian_text(translation, is_translation=True)
    
    return clean_translit, clean_translation


# Example usage with your data:
def preprocess_csv_data(df):
    """
    Preprocess a DataFrame containing 'transliteration' and 'translation' columns.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame with at least 'transliteration' and 'translation' columns
    
    Returns:
    --------
    pandas.DataFrame : DataFrame with new 'clean_transliteration' and 'clean_translation' columns
    """
    import pandas as pd
    
    results = []
    for idx, row in df.iterrows():
        try:
            clean_translit, clean_trans = preprocess_dataset_entry(
                row['transliteration'], 
                row['translation']
            )
            results.append({
                'oare_id': row['oare_id'],
                'clean_transliteration': clean_translit,
                'clean_translation': clean_trans,
                'original_transliteration': row['transliteration'],
                'original_translation': row['translation']
            })
        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            # Keep original if preprocessing fails
            results.append({
                'oare_id': row['oare_id'],
                'clean_transliteration': row['transliteration'],
                'clean_translation': row['translation'],
                'original_transliteration': row['transliteration'],
                'original_translation': row['translation']
            })
    
    return pd.DataFrame(results)

In [7]:
data_pre_processed_pd = preprocess_csv_data(data_pd)

In [8]:
data_pre_processed_pd

,oare_id,clean_transliteration,clean_translation,original_transliteration,original_translation
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lu2m-a-šur DUMU ṣi2-la2-dIM KIŠ...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ...",KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ..."
1,0064939c-59b9-4448-a63d-34612af0a1b5,TÚG ša qa2-tim i-tur4-DINGIR il5-qe2,Itūr-ilī has received one te <gap> tile of ord...,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...
2,0073f2c0-524c-4bbf-915a-8c1772a4fb98,TÚG u-la i-di2-na-ku-um i-tu3-ra-ma GÍN KÙBABBAR,<big_gap> he did not give you a te <gap> tile....,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,... he did not give you a textile. He returned...
3,009fb838-8038-42bc-ad34-5f795b3840ee,KIŠIB šu-dENLÍL DUMU šu-ku-bi-im KIŠIB ṣi2-lu-...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū...",KIŠIB šu-(d)EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū..."
4,00aa1c55-c80c-4346-a159-73ad43ab0ff7,um-ma šu-ku-tum-ma a-na IŠTAR-la2-ma-si2 u3 ni...,From Šukkutum to Ištar-lamassī and Nitahšušar....,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...
...,...,...,...,...,...
1556,ff3208e4-8ab8-4368-b4df-7b80afa5bc32,um-ma en-nam-a-šur-ma a-na a-la-hi-im qi2-bi-m...,From Ennam-Aššur to Ali-ahum. Here men have re...,um-ma en-nam-a-šur-ma a-na a-la-ḫi-im qí-bi-ma...,From Ennam-Aššur to Ali-ahum: Here 2 men have ...
1557,ff43a284-3d67-4238-8b4a-9b6cb7531e0a,ma-na KÙBABBAR ṣa-ru-pa2-am i-na ši2-im SÍGHIA...,Ilī-ašrannī son of Sukkalliya has received min...,3 ma-na KÙ.BABBAR ṣa-ru-pá-am i-na ší-im SÍG.Ḫ...,Ilī-ašrannī son of Sukkalliya has received 3 m...
1558,ff5747a4-af8a-4100-a906-a2660ae72606,ša-lim-a-šu3r a-na a-mur-IŠTAR u2-ṭa2-hi-ni-a-...,Šalim-Aššur made us approach Amur-Ištar and Ša...,ša-lim-a-šùr a-na a-mur-IŠTAR ú-ṭá-ḫi-ni-a-tí-...,Šalim-Aššur made us approach Amur-Ištar and Ša...
1559,ff777871-97ce-4bfc-bdfb-73352868944d,a-na en-nam-a-šu3r qi2-bi-ma um-ma IŠTAR-ra-bi...,To Ennam-Aššur from Ištar-rabiʾat. With respec...,a-na en-nam-a-šùr qí-bi-ma um-ma IŠTAR-ra-bi₄-...,To Ennam-Aššur from Ištar-rabiʾat: With respec...


In [9]:
def preprocess(batch):
    # Add task prefix
    inputs = ["translate Akkadian to English: " + x for x in batch["clean_transliteration"]]
    model_inputs = tokenizer(
        inputs,
        truncation=True,
        #padding="max_length",
        max_length=512
    )

    # Tokenize labels
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["original_translation"],
            truncation=True,
            #padding="max_length",
            max_length=256
        )["input_ids"]

    # Replace pad tokens with -100
    labels = [[(tok if tok != tokenizer.pad_token_id else -100) for tok in l] for l in labels]

    model_inputs["labels"] = labels
    return model_inputs

dataset = Dataset.from_pandas(data_pre_processed_pd) 

tokenized = dataset.map(preprocess, batched=True)

Map:   0%|          | 0/1561 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [10]:
bleu = evaluate.load("sacrebleu")

In [11]:
EPOCH = 40

training_args = TrainingArguments(
    output_dir="./byt5-akkadian",
    eval_strategy="steps",
    save_strategy="steps",
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    weight_decay=0.01,
    fp16=False,
    logging_steps=176,
    save_steps=176*EPOCH/2,
    eval_steps=176,
    max_grad_norm=1.0,
    num_train_epochs=EPOCH,
    gradient_checkpointing=True,
    report_to="none"
)

In [12]:
tokenized = tokenized.train_test_split(test_size=0.1, seed=42)

tokenized

DatasetDict({
    train: Dataset({
        features: ['oare_id', 'clean_transliteration', 'clean_translation', 'original_transliteration', 'original_translation', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1404
    })
    test: Dataset({
        features: ['oare_id', 'clean_transliteration', 'clean_translation', 'original_transliteration', 'original_translation', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 157
    })
})

In [13]:
def compute_metrics(eval_preds):
    predictions, labels = eval_preds

    # If predictions are logits, extract and convert
    if isinstance(predictions, tuple):
        # Some seq2seq models return (logits, _) in a tuple
        predictions = predictions[0]

    # If logits, take argmax along vocab axis
    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    # Replace -100 in labels with pad_token_id
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # Decode token IDs
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # BLEU expects list of list of references
    references = [[l] for l in decoded_labels]

    results = bleu.compute(predictions=decoded_preds, references=references)
    return {"bleu": results["score"]}

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

/tmp/ipykernel_25/1356027557.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss,Validation Loss,Bleu
176,0.286800,0.491309,37.814134
352,0.288600,0.486441,38.094464
528,0.284900,0.482912,38.278850
704,0.283300,0.489354,38.229510
880,0.280800,0.495644,38.589312
1056,0.280500,0.490378,38.667692
1232,0.278900,0.491516,38.970380
1408,0.276300,0.488575,38.703013
1584,0.276200,0.500583,38.534615
1760,0.274600,0.499402,38.591794


TrainOutput(global_step=7040, training_loss=0.2702554870735515, metrics={'train_runtime': 19621.2388, 'train_samples_per_second': 2.862, 'train_steps_per_second': 0.359, 'total_flos': 7.439940192927744e+16, 'train_loss': 0.2702554870735515, 'epoch': 40.0})

In [15]:
#data_pre_processed_pd.to_csv('data_pre_processed.csv', index=False)